In [23]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping

data = pd.read_csv("English2Hindi.csv").dropna(subset=["English", "Hindi"])
data["English"] = data["English"].astype(str)
data["Hindi"] = data["Hindi"].astype(str)

# Keep short sentences to make training faster
data = data[
    (data["English"].str.split().str.len() <= 20) &
    (data["Hindi"].str.split().str.len() <= 20)
].reset_index(drop=True)

In [24]:
# Build word dictionaries
eng_words = sorted(set(" ".join(data["English"]).split()))
hin_words = sorted(set(" ".join(data["Hindi"]).split()))

eng_to_id = {word: i + 1 for i, word in enumerate(eng_words)}
hin_to_id = {word: i + 1 for i, word in enumerate(hin_words)}
id_to_hin = {i: word for word, i in hin_to_id.items()}

max_eng_len = int(data["English"].str.split().str.len().max())
max_hin_len = int(data["Hindi"].str.split().str.len().max())
eng_vocab_size = len(eng_to_id) + 1  # 0 is padding
hin_vocab_size = len(hin_to_id) + 1

X_train, X_test, y_train, y_test = train_test_split(
    data["English"], data["Hindi"], test_size=0.2, random_state=42
)


In [25]:
# Convert sentences into padded integer arrays
def encode_text(text, word_to_id, max_len):
    result = np.zeros(max_len, dtype="int32")
    for i, word in enumerate(text.split()[:max_len]):
        result[i] = word_to_id.get(word, 0)
    return result

def make_arrays(X, y):
    enc = np.array([encode_text(s, eng_to_id, max_eng_len) for s in X])
    dec_in = np.array([encode_text(s, hin_to_id, max_hin_len) for s in y])
    dec_out = np.zeros((len(y), max_hin_len), dtype="int32")
    for i, sentence in enumerate(y):
        ids = [hin_to_id.get(w, 0) for w in sentence.split()[:max_hin_len]]
        if len(ids) > 1:
            dec_out[i, :len(ids)-1] = ids[1:]
    return enc, dec_in, dec_out

Xtr, Ytr, Ytr_out = make_arrays(X_train.tolist(), y_train.tolist())
Xte, Yte, Yte_out = make_arrays(X_test.tolist(), y_test.tolist())


In [26]:
# Encoder-decoder model
latent_dim = 128

enc_input = Input(shape=(max_eng_len,))
enc_embedding = Embedding(eng_vocab_size, latent_dim, mask_zero=True)(enc_input)
_, enc_h, enc_c = LSTM(latent_dim, return_state=True)(enc_embedding)
enc_states = [enc_h, enc_c]

dec_input = Input(shape=(max_hin_len,))
dec_embedding_layer = Embedding(hin_vocab_size, latent_dim, mask_zero=True)
dec_embedding = dec_embedding_layer(dec_input)
dec_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
dec_output, _, _ = dec_lstm(dec_embedding, initial_state=enc_states)
dec_dense = Dense(hin_vocab_size, activation="softmax")
dec_output = dec_dense(dec_output)

model = Model([enc_input, dec_input], dec_output)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
model.summary()


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_4 (InputLayer)    │ (None, 20)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ input_layer_5 (InputLayer)    │ (None, 20)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding_4 (Embedding)       │ (None, 20, 128)           │       8,889,984 │ input_layer_4[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ not_equal_4 (NotEqual)        │ (None, 20)                │               0 │ input_layer_4[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding_5 (Embedding)       │ (None, 20, 128)           │       6,414,336 │ input_layer_5[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_4 (LSTM)                 │ [(None, 128), (None,      │         131,584 │ embedding_4[0][0],         │
│                               │ 128), (None, 128)]        │                 │ not_equal_4[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ lstm_5 (LSTM)                 │ [(None, 20, 128), (None,  │         131,584 │ embedding_5[0][0],         │
│                               │ 128), (None, 128)]        │                 │ lstm_4[0][1], lstm_4[0][2] │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_2 (Dense)               │ (None, 20, 50112)         │       6,464,448 │ lstm_5[0][0]               │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 22,031,936 (84.05 MB)

 Trainable params: 22,031,936 (84.05 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Train only as long as validation loss keeps improving
early_stop = EarlyStopping(
    monitor="val_loss", patience=2, restore_best_weights=True
)

history = model.fit(
    [Xtr, Ytr], Ytr_out,
    validation_data=([Xte, Yte], Yte_out),
    batch_size=128,
    epochs=10,                 # maximum; EarlyStopping may stop sooner
    callbacks=[early_stop],
    verbose=1
)


Epoch 1/10
537/537 ━━━━━━━━━━━━━━━━━━━━ 709s 1s/step - loss: 7.1862 - val_loss: 6.6287
Epoch 2/10
537/537 ━━━━━━━━━━━━━━━━━━━━ 621s 1s/step - loss: 6.3407 - val_loss: 6.2535
Epoch 3/10
537/537 ━━━━━━━━━━━━━━━━━━━━ 618s 1s/step - loss: 5.9008 - val_loss: 5.9555
Epoch 4/10
537/537 ━━━━━━━━━━━━━━━━━━━━ 617s 1s/step - loss: 5.5565 - val_loss: 5.7773
Epoch 5/10
537/537 ━━━━━━━━━━━━━━━━━━━━ 621s 1s/step - loss: 5.2884 - val_loss: 5.6418
Epoch 6/10
318/537 ━━━━━━━━━━━━━━━━━━━━ 4:27 1s/step - loss: 5.0653

In [21]:
# Inference models
encoder_model = Model(enc_input, enc_states)

state_h = Input(shape=(latent_dim,))
state_c = Input(shape=(latent_dim,))
decoder_states = [state_h, state_c]

step_embedding = dec_embedding_layer(dec_input)
step_output, out_h, out_c = dec_lstm(
    step_embedding, initial_state=decoder_states
)
step_output = dec_dense(step_output)
decoder_model = Model(
    [dec_input, state_h, state_c],
    [step_output, out_h, out_c]
)

def translate(sentence):
    encoded = encode_text(sentence, eng_to_id, max_eng_len)[None, :]
    h, c = encoder_model.predict(encoded, verbose=0)
    result = []

    # Start from the first Hindi word ID available; this dataset version
    # does not define explicit START_/END_ tokens.
    current_id = 0
    for _ in range(max_hin_len):
        dec = np.zeros((1, max_hin_len), dtype="int32")
        dec[0, 0] = current_id
        probs, h, c = decoder_model.predict([dec, h, c], verbose=0)
        current_id = int(np.argmax(probs[0, 0]))
        if current_id == 0:
            break
        result.append(id_to_hin.get(current_id, ""))
    return " ".join(result)

sample = X_test.iloc[0]
print("English:", sample)
print("Predicted Hindi:", translate(sample))
print("Actual Hindi:", y_test.iloc[0])


In [ ]:
def translate(inp_seq):
    # Encode the input as state vectors.
    states_value = enc_model.predict(inp_seq)
    # Generate empty target sequence of length 1.
    tar_seq = np.zeros((1,1))
    # Populate the first character of target sequence with the start character.
    tar_seq[0, 0] = tar_tok_idx['START_']

    # Sampling loop for a batch of sequences
    # (to simplify, here we assume a batch of size 1).
    stop_cond = False
    dec_sen = ''
    while not stop_cond:
        output_toks, h, c = dec_model.predict([tar_seq] + states_value)

        # Sample a token
        sampled_tok_idx = np.argmax(output_toks[0, -1, :])
        sampled_char = rev_tar_char_idx[sampled_tok_idx]
        dec_sen += ' '+sampled_char

        # Exit condition: either hit max length
        # or find stop character.
        if (sampled_char == '_END' or
           len(dec_sen) > 50):
            stop_cond = True

        # Update the target sequence (of length 1).
        tar_seq = np.zeros((1,1))
        tar_seq[0, 0] = sampled_tok_idx

        # Update states
        states_value = [h, c]

    return dec_sen

train_gen = generate_batch(X_train, y_train, batch_size = 1)
k=0
(inp_seq, actual_output), _ = next(train_gen)
hin_sen = translate(inp_seq)
print(f'''Input English sentence: {X_train[k:k+1].values[0]}\n
          Predicted Hindi Translation: {hin_sen[:-4]}\n
          Actual Hindi Translation: {y_train[k:k+1].values[0][6:-4]}''')

1/1 [==============================] - 0s 19ms/step
Input English sentence: which is a pity but in india every other sport

Actual Hindi Translation:  जिसपे हमें तरस आती है लेकिन भारत में हर खेल 

Predicted Hindi Translation:  जिसपे हमें तरस आती है कि भारत में एक व्यक्ति 
